# Fake News Detection with NLP

This notebook builds a reproducible Natural Language Processing workflow for fake news detection. It loads a labeled news dataset, explores category and text patterns, trains a TF-IDF based classifier, evaluates the model, and exports a reusable pipeline for downstream applications.

## Project Goals

- Prepare a clean, labeled dataset of true and fake news articles.
- Compare class balance, article categories, and text length patterns.
- Train a deterministic baseline classifier with scikit-learn.
- Evaluate model quality with accuracy, balanced accuracy, a classification report, and a confusion matrix.
- Inspect the words and phrases that most influence the predictions.
- Save the trained model pipeline for production handoff or experimentation.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
SAMPLE_SIZE = int(os.getenv("SAMPLE_SIZE", "0")) or None

## Data Source

The project uses the public fake news dataset provided for the NLP course. The archive contains two files:

- `True.csv`: verified news articles.
- `Fake.csv`: fake news articles.

The notebook first looks for local data under `data/fake_news/` or the legacy course folder `fake_news/`. If the files are missing, it downloads the archive automatically.

In [ ]:
DATA_URL = "https://proai-datasets.s3.eu-west-3.amazonaws.com/fake_news.zip"
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "fake_news"
LEGACY_RAW_DIR = Path("fake_news")
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / "fake_news_tfidf_logistic_regression.joblib"


def has_expected_files(directory: Path) -> bool:
    return (directory / "True.csv").exists() and (directory / "Fake.csv").exists()


def ensure_dataset() -> Path:
    if has_expected_files(RAW_DIR):
        return RAW_DIR
    if has_expected_files(LEGACY_RAW_DIR):
        return LEGACY_RAW_DIR

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = DATA_DIR / "fake_news.zip"
    if not archive_path.exists():
        print(f"Downloading dataset to {archive_path}...")
        urlretrieve(DATA_URL, archive_path)

    with ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

    candidate_dirs = [RAW_DIR, DATA_DIR, LEGACY_RAW_DIR]
    candidate_dirs.extend(path for path in DATA_DIR.rglob("*") if path.is_dir())
    for candidate in candidate_dirs:
        if has_expected_files(candidate):
            return candidate

    raise FileNotFoundError("Could not find True.csv and Fake.csv after extracting the dataset.")


raw_dir = ensure_dataset()
raw_dir

## Load and Label the Articles

The original files use the same schema, so the only label engineering required is adding a numeric target and a human-readable label.

In [ ]:
def load_articles(file_path: Path, label: int, label_name: str) -> pd.DataFrame:
    return pd.read_csv(file_path).assign(label=label, label_name=label_name)


true_articles = load_articles(raw_dir / "True.csv", label=0, label_name="true")
fake_articles = load_articles(raw_dir / "Fake.csv", label=1, label_name="fake")

news = pd.concat([true_articles, fake_articles], ignore_index=True)
news["title"] = news["title"].fillna("")
news["text"] = news["text"].fillna("")
news["content"] = (news["title"] + ". " + news["text"]).str.strip()
news = news.drop_duplicates(subset=["title", "text"]).reset_index(drop=True)

if SAMPLE_SIZE is not None:
    news = (
        news.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), SAMPLE_SIZE // 2), random_state=RANDOM_STATE))
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

news.head()

In [ ]:
dataset_summary = (
    news.groupby("label_name")
    .agg(articles=("label", "size"), subjects=("subject", "nunique"))
    .reset_index()
)

dataset_summary

## Exploratory Data Analysis

Before training the classifier, check whether the classes are balanced and whether article categories differ between true and fake news.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

news["label_name"].value_counts().reindex(["true", "fake"]).plot(
    kind="bar",
    ax=axes[0],
    color=["#4C78A8", "#E45756"],
)
axes[0].set_title("Class balance")
axes[0].set_xlabel("")
axes[0].set_ylabel("Articles")
axes[0].tick_params(axis="x", rotation=0)

subject_by_label = pd.crosstab(news["subject"], news["label_name"]).sort_values("fake", ascending=True)
subject_by_label.plot(kind="barh", stacked=True, ax=axes[1], color=["#E45756", "#4C78A8"])
axes[1].set_title("Articles by subject and label")
axes[1].set_xlabel("Articles")
axes[1].set_ylabel("")

plt.tight_layout()

In [ ]:
news["word_count"] = news["content"].str.split().str.len()

length_summary = news.groupby("label_name")["word_count"].describe().round(1)
length_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label_name, color in [("true", "#4C78A8"), ("fake", "#E45756")]:
    values = news.loc[news["label_name"] == label_name, "word_count"].clip(upper=2_000)
    ax.hist(values, bins=40, alpha=0.55, label=label_name, color=color)

ax.set_title("Article length distribution")
ax.set_xlabel("Word count, clipped at 2,000")
ax.set_ylabel("Articles")
ax.legend()
plt.tight_layout()

## Train/Test Split

The split is stratified so both classes keep the same proportion in the training and test sets.

In [ ]:
X = news["content"]
y = news["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape

## Baseline NLP Classifier

The model uses a compact, production-friendly scikit-learn pipeline:

1. `TfidfVectorizer` converts article text into weighted word and bigram features.
2. `LogisticRegression` learns a linear decision boundary that remains easy to inspect.

This is a strong baseline for text classification and is much easier to deploy than a notebook-only sequence of manual preprocessing steps.

In [ ]:
model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.90,
                max_features=100_000,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1_000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

metrics = pd.DataFrame(
    {
        "metric": ["accuracy", "balanced_accuracy"],
        "value": [
            accuracy_score(y_test, predictions),
            balanced_accuracy_score(y_test, predictions),
        ],
    }
)

metrics

In [ ]:
print(classification_report(y_test, predictions, target_names=["true", "fake"]))

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["true", "fake"],
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("Confusion matrix")
plt.tight_layout()

## Save the Model

The exported pipeline includes both the vectorizer and classifier, so it can transform raw article text directly at inference time.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
MODEL_PATH

## Interpret the Classifier

Because logistic regression is linear, the feature weights show which terms most strongly push predictions toward `fake` or `true`.

In [ ]:
def get_top_features(pipeline: Pipeline, n: int = 20) -> tuple[pd.DataFrame, pd.DataFrame]:
    vectorizer = pipeline.named_steps["tfidf"]
    classifier = pipeline.named_steps["classifier"]

    feature_names = np.array(vectorizer.get_feature_names_out())
    coefficients = classifier.coef_[0]

    fake_indices = np.argsort(coefficients)[-n:][::-1]
    true_indices = np.argsort(coefficients)[:n]

    top_fake = pd.DataFrame(
        {"feature": feature_names[fake_indices], "weight": coefficients[fake_indices]}
    )
    top_true = pd.DataFrame(
        {"feature": feature_names[true_indices], "weight": coefficients[true_indices]}
    )
    return top_fake, top_true


top_fake_terms, top_true_terms = get_top_features(model, n=20)

display(top_fake_terms.head(10))
display(top_true_terms.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_fake_terms.sort_values("weight").plot(
    x="feature",
    y="weight",
    kind="barh",
    ax=axes[0],
    color="#E45756",
    legend=False,
)
axes[0].set_title("Top fake-news indicators")
axes[0].set_xlabel("Coefficient weight")
axes[0].set_ylabel("")

top_true_terms.assign(abs_weight=top_true_terms["weight"].abs()).sort_values("abs_weight").plot(
    x="feature",
    y="abs_weight",
    kind="barh",
    ax=axes[1],
    color="#4C78A8",
    legend=False,
)
axes[1].set_title("Top true-news indicators")
axes[1].set_xlabel("Absolute coefficient weight")
axes[1].set_ylabel("")

plt.tight_layout()

## Topic Snapshot for Fake News

The original project explored topics in fake news articles. This version keeps that idea with an NMF topic model built on TF-IDF features, which avoids extra NLP dependencies while still surfacing recurring themes.

In [ ]:
fake_content = news.loc[news["label_name"] == "fake", "content"]

# Keep topic modeling robust for both the full dataset and small smoke-test samples.
topic_min_df = 10 if len(fake_content) >= 1_000 else 2
topic_max_df = 0.95 if len(fake_content) >= 1_000 else 1.0

topic_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    strip_accents="unicode",
    min_df=topic_min_df,
    max_df=topic_max_df,
    max_features=5_000,
)

topic_matrix = topic_vectorizer.fit_transform(fake_content)
n_topics = min(5, topic_matrix.shape[0], topic_matrix.shape[1])

nmf_model = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=RANDOM_STATE,
    max_iter=300,
)
nmf_model.fit(topic_matrix)

topic_terms = np.array(topic_vectorizer.get_feature_names_out())
topics = []
for topic_number, topic_weights in enumerate(nmf_model.components_, start=1):
    top_term_indices = topic_weights.argsort()[-10:][::-1]
    topics.append(
        {
            "topic": f"Topic {topic_number}",
            "top_terms": ", ".join(topic_terms[top_term_indices]),
        }
    )

pd.DataFrame(topics)

## Try the Model on New Text

The model predicts labels from raw article text. The examples below are intentionally short; real usage should pass a complete title and article body.

In [ ]:
example_articles = pd.Series(
    [
        "Reuters reports that the central bank left interest rates unchanged after its scheduled policy meeting.",
        "A viral post claims a secret celebrity-backed cure was hidden by officials and will change everything overnight.",
    ]
)

example_predictions = model.predict(example_articles)
example_probabilities = model.predict_proba(example_articles)[:, 1]

pd.DataFrame(
    {
        "text": example_articles,
        "predicted_label": np.where(example_predictions == 1, "fake", "true"),
        "fake_probability": example_probabilities.round(3),
    }
)

## Limitations and Next Steps

This model is a text classifier, not a fact-checking system. It can learn linguistic style, source patterns, and dataset-specific artifacts, so it should be validated on fresh articles before any real-world deployment.

Recommended next steps:

- Evaluate on newer news data from sources outside the training dataset.
- Add calibration checks before using predicted probabilities in a product.
- Compare this baseline with transformer embeddings or fine-tuned language models.
- Wrap the exported pipeline in an API and monitor drift over time.

## Conclusion

The final pipeline provides a reproducible and interpretable baseline for fake news detection. It is simple to run, easy to inspect, and ready to be versioned as a GitHub project.